# Triagegeist — Honest, Leakage-Controlled ESI Triage Prediction

**A clinical decision-support pipeline for emergency-department triage acuity (ESI 1–5).**

This notebook predicts triage acuity from structured intake data (vitals, demographics,
clinical context) and free-text chief complaints. Its central contribution is *methodological
honesty*: we show that the chief-complaint text in this synthetic dataset **encodes the label**,
inflating scores to ~0.99 through leakage, and we deliberately build an honest model that
reports a realistic ~0.91 weighted kappa instead.

**What this notebook demonstrates**
1. A two-part **leakage screen** — obvious post-triage columns *and* a subtle text-encoding leak.
2. A reproducible **leakage demonstration**: adding text features moves an identical model from ~0.91 to ~0.999.
3. An **honest model** (LightGBM via AutoGluon) reporting ~0.911 OOF linear-weighted kappa.
4. **Dual interpretability** (SHAP + glass-box EBM) confirming the model relies on physiology, not text.
5. A **cost-sensitive undertriage analysis** — the safety metrics that matter clinically.

> **Note on scoring:** A perfect-looking score on this data is a red flag, not a goal. We argue the
> honest performance ceiling is ~0.91, bounded by irreducible label noise, and that scores near
> 0.99 reflect leakage of the data-generating process rather than clinical skill.


## Section 0 — Setup & Environment

Installs and imports the libraries used throughout (LightGBM, SHAP, `interpret`/EBM, AutoGluon),
then prints the resolved versions and whether a GPU is present. The pipeline is **CPU-only** — no
GPU is required to reproduce any result below.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 0 — Setup & Installs
# Triagegeist: cost-sensitive, leakage-controlled ESI triage prediction.
# CPU-only pipeline. Installs only what the notebook uses; most packages are
# preinstalled on Kaggle, so this step is typically near-instant.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, sys, importlib, warnings, os
warnings.filterwarnings('ignore')

def pip_install(args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

# --- Core modelling + interpretability (install only if missing) ---
for pkg in ['lightgbm', 'shap']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        pip_install([pkg])

# --- EBM (glass-box model) ---
try:
    import interpret  # noqa
except ImportError:
    pip_install(['interpret'])

# --- AutoGluon (tabular only, to keep it lean) ---
try:
    import autogluon.tabular  # noqa
except ImportError:
    pip_install(['autogluon.tabular'])

# --- boto3 (only used by the optional AWS Bedrock narrative layer in Section 9) ---
try:
    import boto3  # noqa
except ImportError:
    pip_install(['boto3'])

# ── Report what we actually have ──────────────────────────────────────────
def ver(name, import_as=None):
    try:
        m = importlib.import_module(import_as or name)
        print(f"  ✓ {name:18s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  ✗ {name:18s} NOT AVAILABLE — {e}")

print("Environment check:")
ver('numpy'); ver('pandas'); ver('lightgbm'); ver('shap')
ver('interpret'); ver('autogluon', 'autogluon.tabular'); ver('boto3')
print("\nSetup complete (CPU-only).")

## Section 1 — Data Load & Join

Loads the five competition files and joins patient history and chief-complaint text onto the
train/test tables via `patient_id`. We confirm 100% join coverage, review the class distribution
(ESI-3 is the plurality at ~36%; ESI-1 is rare at ~4%), and flag the columns that exist only in
train — candidate leakage to investigate next.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — Data Load & Join
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
pd.set_option('display.max_columns', None)

SEED = 42
np.random.seed(SEED)

# Kaggle paths (confirmed)
DATA = '/kaggle/input/competitions/triagegeist'

train_raw  = pd.read_csv(f'{DATA}/train.csv')
test_raw   = pd.read_csv(f'{DATA}/test.csv')
cc         = pd.read_csv(f'{DATA}/chief_complaints.csv')
ph         = pd.read_csv(f'{DATA}/patient_history.csv')
sample_sub = pd.read_csv(f'{DATA}/sample_submission.csv')

TARGET = 'triage_acuity'

print("Shapes:")
for n, d in [('train', train_raw), ('test', test_raw),
             ('chief_complaints', cc), ('patient_history', ph),
             ('sample_submission', sample_sub)]:
    print(f"  {n:18s} {d.shape}")

# Sanity: every test/train id present in cc and ph
def coverage(name, df):
    in_cc = df['patient_id'].isin(cc['patient_id']).mean()
    in_ph = df['patient_id'].isin(ph['patient_id']).mean()
    print(f"  {name}: cc match {in_cc:.1%}, ph match {in_ph:.1%}")
print("\nJoin coverage:")
coverage('train', train_raw); coverage('test', test_raw)

print("\nTarget distribution (train):")
print((train_raw[TARGET].value_counts(normalize=True).sort_index() * 100
       ).round(1).astype(str).add(' %').to_string())

print(f"\nColumns only in train (potential outcomes/leakage): "
      f"{sorted(set(train_raw.columns) - set(test_raw.columns))}")

**Output read:** All joins match at 100%. The target is imbalanced toward the middle of the scale,
and three columns appear only in train — `disposition`, `ed_los_hours`, and the target itself.
The first two describe what happened *after* triage, so they are leakage candidates examined in Section 2.


## Section 2 — Leakage Screen *(headline finding)*

Before modelling, we audit for label leakage and find **two kinds**:

- **Part A — obvious:** `ed_los_hours` and `disposition` are post-triage outcomes, unknown at triage time.
- **Part B — subtle and severe:** the free-text chief complaint *encodes* the label. We test this three ways:
  1. a model trained on **text alone** (no vitals),
  2. how often identical complaint strings map to a single acuity,
  3. how much *genuine* clinical signal the text carries (keyword flags only).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — Leakage Screen
#   Before modelling, we audit the data for label leakage. We find TWO kinds:
#   (A) obvious post-triage outcome columns, and (B) a subtle, severe one — the
#   free-text chief complaint encodes the label almost perfectly. This screen
#   shapes every downstream modelling choice.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import cohen_kappa_score, accuracy_score
import lightgbm as lgb

SEED = 42
y = train_raw['triage_acuity'].values
txt = (train_raw.merge(cc[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
       ['chief_complaint_raw'].fillna('').astype(str))

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
quick = dict(objective='multiclass', num_class=5, n_estimators=400, learning_rate=0.05,
             num_leaves=127, random_state=SEED, n_jobs=-1, verbose=-1)

print("="*65)
print("PART A — Obvious leakage: post-triage outcome columns")
print("="*65)
# These exist in train only and describe what happened AFTER triage.
print(f"  ed_los_hours corr with acuity : {train_raw['ed_los_hours'].corr(train_raw['triage_acuity']):.3f}")
disp = train_raw.groupby('disposition')['triage_acuity'].mean()
print(f"  disposition mean-acuity spread: {disp.min():.2f} → {disp.max():.2f}")
print("  → ed_los_hours, disposition are unknown at triage time. Will be dropped.\n")

print("="*65)
print("PART B — Subtle leakage: does the chief-complaint TEXT encode the label?")
print("="*65)

# B1: text ONLY (TF-IDF + SVD), no vitals at all
vec  = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=4000, sublinear_tf=True)
Xsvd = TruncatedSVD(32, random_state=SEED).fit_transform(vec.fit_transform(txt))
oof_txt = cross_val_predict(lgb.LGBMClassifier(**quick), Xsvd, y-1, cv=skf, method='predict') + 1
k_txt = cohen_kappa_score(y, oof_txt, weights='linear')
print(f"  B1) Chief-complaint TEXT ONLY (no vitals)  → kappa = {k_txt:.4f}, acc = {accuracy_score(y,oof_txt):.4f}")

# B2: lookup-table fingerprint — do identical complaints map to one acuity?
norm = txt.str.lower().str.replace(r'[^a-z ]','',regex=True).str.strip()
g = pd.DataFrame({'t':norm,'y':y}).groupby('t')['y'].agg(['nunique','count'])
g = g[g['count'] >= 5]
pure = (g['nunique'] == 1).mean()
print(f"  B2) Of complaint strings seen ≥5×, fraction mapping to ONE acuity: {pure:.1%}")

# B3: honest clinical keyword signal only
t = txt.str.lower()
kw = pd.DataFrame({
    'kw_crit': t.str.contains(r'arrest|unrespons|not breathing|anaphylaxis|stroke|seizure|overdose', regex=True).astype(int),
    'kw_high': t.str.contains(r'chest pain|short.*breath|severe|sudden|syncope|altered', regex=True).astype(int),
    'kw_low' : t.str.contains(r'\bmild\b|minor|routine|chronic|refill|rash', regex=True).astype(int),
})
oof_kw = cross_val_predict(lgb.LGBMClassifier(**quick), kw.values, y-1, cv=skf, method='predict') + 1
k_kw = cohen_kappa_score(y, oof_kw, weights='linear')
print(f"  B3) Honest clinical KEYWORD flags only     → kappa = {k_kw:.4f}, acc = {accuracy_score(y,oof_kw):.4f}")

print("\n" + "-"*65)
print(f"  VERDICT: text alone predicts acuity at {k_txt:.3f}, and {pure:.1%} of repeated")
print(f"  complaints map to a single acuity — the synthetic text ENCODES the label.")
print(f"  Genuine clinical text signal is only {k_kw:.3f}. The gap ({k_txt-k_kw:.3f}) is leakage.")
print(f"  → We EXCLUDE high-dimensional text features; use only interpretable keyword flags.")
print("-"*65)

# Stash for the writeup / later sections
LEAK_OUTCOME = ['ed_los_hours', 'disposition']
leakage_results = {'text_only_kappa': k_txt, 'lookup_purity': pure, 'keyword_kappa': k_kw}

**Output read — this is the central result.** Chief-complaint text *alone*, with no vitals,
predicts acuity at **≈0.999 kappa**, and **99.6%** of complaint strings seen five or more times map to
exactly one acuity level — the text behaves as a near-perfect lookup table for the label. Yet the
genuine clinical signal in the text (interpretable keyword flags) yields only **≈0.27**. The gap
(~0.73 kappa) is leakage of the data-generating process, not clinical information. **Consequence:**
we exclude high-dimensional text features from the model and use only interpretable keyword flags.


## Section 3 — Feature Engineering (honest, leakage-free)

We engineer features using only information available at triage time: vitals, demographics, clinical
context, comorbidity flags, and — from the chief complaint — **only interpretable keyword flags**
(critical / acute / breathing / chest-pain / mild), never TF-IDF. We drop the post-triage outcome
columns and the rater/site identifiers, and encode `pain_score = -1` as "not recorded" rather than a
true zero. Missingness itself is retained as signal.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — Feature Engineering (honest, leakage-free)
#   Per the Section 2 screen, we use ONLY interpretable clinical keyword flags
#   from the chief complaint — NOT high-dimensional TF-IDF, which leaks the
#   label. We drop the post-triage outcome columns and rater/site identifiers.
# ══════════════════════════════════════════════════════════════════════════
from pandas.api.types import is_numeric_dtype

TARGET = 'triage_acuity'
LEAK_OUTCOME  = ['ed_los_hours', 'disposition']   # post-triage; unknown at triage time
LEAK_IDENTITY = ['triage_nurse_id', 'site_id']    # rater/site identity (kept out of model)

KW_PATTERNS = {
    'kw_critical' : r'shock|arrest|stroke|stemi|sepsis|haemorrhage|hemorrhage|unrespons',
    'kw_acute'    : r'\bacute\b|severe|worst|sudden',
    'kw_breath'   : r'breath|dyspnea|dyspnoea|sob\b',
    'kw_chestpain': r'chest pain',
    'kw_mild'     : r'\bmild\b|routine|refill|follow.?up|review',
}
VITALS_MISSABLE = ['systolic_bp', 'respiratory_rate', 'temperature_c', 'spo2']

def engineer(df):
    """Leakage-free feature engineering. Target is never used. Safe for train and test."""
    df = (df.merge(ph, on='patient_id', how='left')
            .merge(cc[['patient_id','chief_complaint_raw']], on='patient_id', how='left'))

    # Chief-complaint: ONLY interpretable keyword flags (no TF-IDF — see Section 2)
    txt = (df['chief_complaint_raw'].fillna('')
             .str.replace('\uff0c', ',', regex=False).str.lower())   # normalise fullwidth comma
    for name, pat in KW_PATTERNS.items():
        df[name] = txt.str.contains(pat, regex=True).astype(int)
    df['cc_len'] = txt.str.len()

    # pain_score: -1 means "not recorded", not a true 0
    df['pain_not_recorded'] = (df['pain_score'] == -1).astype(int)
    df['pain_score'] = df['pain_score'].replace(-1, np.nan)

    # Missingness as signal (low-acuity patients skip vitals)
    for c in VITALS_MISSABLE:
        df[c + '_miss'] = df[c].isna().astype(int)
    df['vitals_miss_n'] = df[[c + '_miss' for c in VITALS_MISSABLE]].sum(axis=1)

    # Comorbidity burden
    hx_cols = [c for c in df.columns if c.startswith('hx_')]
    df['hx_total'] = df[hx_cols].fillna(0).sum(axis=1)

    return df.drop(columns=['chief_complaint_raw'])

train_eng = engineer(train_raw)
test_eng  = engineer(test_raw)

# Feature set: drop ids, leakage, target, and the two dead features AutoGluon flagged
DEAD = ['spo2_miss', 'hx_total']   # spo2 never missing; hx_total redundant with num_comorbidities
DROP = LEAK_OUTCOME + LEAK_IDENTITY + ['patient_id', TARGET] + DEAD
FEATURES = [c for c in train_eng.columns if c not in DROP]

# Robust categorical detection (non-numeric == categorical; works across pandas versions)
CAT_FEATURES = [c for c in FEATURES if not is_numeric_dtype(train_eng[c])]
for c in CAT_FEATURES:
    train_eng[c] = train_eng[c].astype('category')
    test_eng[c]  = pd.Categorical(test_eng[c], categories=train_eng[c].cat.categories)

X      = train_eng[FEATURES]
y      = train_eng[TARGET]
X_test = test_eng[FEATURES]

print(f"Features: {len(FEATURES)}  ({len(CAT_FEATURES)} categorical, "
      f"{len(FEATURES)-len(CAT_FEATURES)} numeric)")
print(f"Keyword/flag features: {list(KW_PATTERNS) + ['cc_len','pain_not_recorded','vitals_miss_n']}")
print(f"Dropped as leakage: {LEAK_OUTCOME + LEAK_IDENTITY}")
print(f"X {X.shape} | X_test {X_test.shape}")
assert not X.isna().all().any(), "a feature is entirely NaN"
print("Feature engineering complete ✓")

**Output read:** 70 leakage-free features (12 categorical, 58 numeric). The keyword flags replace
high-dimensional text. `ed_los_hours`, `disposition`, `triage_nurse_id`, and `site_id` are excluded
from the model (site is revisited only as an equity *audit* in Section 7).


## Section 4 — Text-Feature Leakage Demonstration *(not submitted)*

A common, intuitive approach is to vectorize the chief complaint with TF-IDF and feed it to the model.
Here we build exactly that, purely to **demonstrate** the leak — this model is **not** submitted. The
ablation holds the model and CV fixed and changes only the feature set, isolating the text as the
cause of the inflation.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 4 — Text-Feature Leakage Demonstration
#   A common, intuitive approach is to vectorize the free-text chief complaint
#   (e.g. TF-IDF) and feed it to the model. On this dataset that yields a very
#   high score — but the gain is leakage: the synthetic text encodes the label.
#   We demonstrate this with an ablation, then exclude text features from the
#   model we actually use. This model is for analysis only; it is NOT submitted.
# ══════════════════════════════════════════════════════════════════════════
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import lightgbm as lgb, numpy as np, pandas as pd, matplotlib.pyplot as plt

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
PARAMS = dict(objective='multiclass', num_class=5, n_estimators=400, learning_rate=0.05,
              num_leaves=127, random_state=SEED, n_jobs=-1, verbose=-1)

def cv_kappa(Xmat, label_offset=1):
    oof = np.zeros((len(Xmat), 5))
    Xdf = pd.DataFrame(Xmat) if not isinstance(Xmat, pd.DataFrame) else Xmat.reset_index(drop=True)
    for tr_i, va_i in skf.split(Xdf, y):
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(Xdf.iloc[tr_i], y.values[tr_i] - label_offset)
        oof[va_i] = m.predict_proba(Xdf.iloc[va_i])
    return cohen_kappa_score(y, oof.argmax(1) + label_offset, weights='linear'), oof

# Honest X as plain numeric (encode categoricals) so we can hstack text onto it
X_num = X.copy()
for c in CAT_FEATURES:
    X_num[c] = X_num[c].cat.codes

# --- Build TF-IDF text features (the ingredient under test) ---
txt_tr = (train_raw.merge(cc[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
          ['chief_complaint_raw'].fillna('').astype(str))
vec  = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=4000, sublinear_tf=True)
Xtf  = vec.fit_transform(txt_tr)
svd  = TruncatedSVD(32, random_state=SEED)
Xsvd = svd.fit_transform(Xtf)

# --- Ablation: identical model, three feature sets ---
k_honest, _ = cv_kappa(X_num)
k_text,   _ = cv_kappa(pd.DataFrame(Xsvd))
X_both = pd.DataFrame(np.hstack([X_num.values, Xsvd]))
k_both,  _ = cv_kappa(X_both)

print("="*60)
print("ABLATION — same LightGBM, same CV, only the feature set changes")
print("="*60)
print(f"  Honest features only (vitals + context) : kappa = {k_honest:.4f}")
print(f"  TF-IDF text only (no vitals)            : kappa = {k_text:.4f}")
print(f"  Honest + TF-IDF                         : kappa = {k_both:.4f}")
print(f"\n  >>> Adding ONLY text features moved kappa {k_honest:.4f} → {k_both:.4f} "
      f"(+{k_both-k_honest:.4f}).")
print(f"  >>> This gain reflects label encoding in the synthetic text, not clinical skill.")

# --- Which TF-IDF words drive the inflated model? ---
m_full = lgb.LGBMClassifier(**PARAMS).fit(X_both, y - 1)
feat_names = list(X_num.columns) + [f'tfidf_{i}' for i in range(Xsvd.shape[1])]
imp = pd.Series(m_full.feature_importances_, index=feat_names).sort_values(ascending=False)

terms = np.array(vec.get_feature_names_out())
comp  = svd.components_
top_tfidf = [f for f in imp.head(20).index if f.startswith('tfidf_')]
print("\nMost influential text components and their top words:")
for f in top_tfidf[:6]:
    idx = int(f.split('_')[1])
    top_words = terms[np.argsort(comp[idx])[::-1][:5]]
    print(f"  {f}: {', '.join(top_words)}")

fig, ax = plt.subplots(figsize=(8,6))
top20 = imp.head(20)[::-1]
colors = ['#3b7dd8' if f.startswith('tfidf_') else '#7e3b9e' for f in top20.index]
top20.plot.barh(ax=ax, color=colors)
ax.set_title('Feature importance with text included (blue = TF-IDF, purple = clinical)',
             fontweight='bold')
ax.set_xlabel('LightGBM importance')
plt.tight_layout(); plt.savefig('text_leakage_importance.png', dpi=120, bbox_inches='tight'); plt.show()

leakage_results.update({'ablation_honest': k_honest, 'ablation_text': k_text, 'ablation_both': k_both})
print("\nThis model is for analysis only. The submitted model (Section 5) excludes text features.")

**Output read:** With identical model and cross-validation, honest features alone score **≈0.91**;
adding the TF-IDF text features lifts it to **≈0.999** — a jump of roughly **+0.09** from the text
alone. The feature-importance chart confirms the mechanism: text components rank *above* SpO₂, NEWS2,
and pain score, and their top words cluster cleanly by acuity tier (e.g. *minor / rash / congestion*
at the low end; *severe / trauma / blunt* at the high end). A model that triages on word patterns
rather than physiology is not clinically valid. **This model is excluded from the submission.**


## Section 5 — Honest Model *(the submitted model)*

Our submitted model uses the leakage-free features from Section 3. A full AutoGluon `best_quality`
search on this data selected a **single LightGBM** (final ensemble weight 1.0) and found that
multi-layer stacking *overfit*; so we fit LightGBM directly with 8-fold bagging — the same result in
a fraction of the runtime. Bagging yields genuine out-of-fold predictions for honest scoring.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 5 — Honest Model (the submitted model)
#   Features: vitals + clinical context + interpretable keyword flags. NO text
#   vectorization (Section 4 showed it leaks). A full AutoGluon best_quality
#   search on this data selected a single LightGBM (ensemble weight 1.0) and
#   found multi-layer stacking caused overfitting — so we fit LightGBM directly
#   with 8-fold bagging. Same performance, a fraction of the runtime.
# ══════════════════════════════════════════════════════════════════════════
from autogluon.tabular import TabularPredictor
from sklearn.metrics import cohen_kappa_score, classification_report
import numpy as np, pandas as pd

ag_train = X.copy(); ag_train[TARGET] = y.values
ag_test  = X_test.copy()

predictor = TabularPredictor(
    label=TARGET,
    problem_type='multiclass',
    eval_metric='accuracy',          # AG has no weighted-kappa metric; we score kappa ourselves
    path='ag_models_honest',
    verbosity=2,
).fit(
    ag_train,
    hyperparameters={
        'GBM': [
            {},                                                       # standard LightGBM (the winner)
            {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}},  # ExtraTrees-style LightGBM
        ],
    },
    num_bag_folds=8,           # 8-fold bagging → true OOF predictions (matches best_quality)
    num_stack_levels=0,        # stacking overfit on this data (confirmed in full run) — off
    dynamic_stacking=False,    # skip the ~15-min DyStack probe; the answer is already known
    ag_args_fit={'num_gpus': 0},
)

# --- Honest OOF kappa (bagged → genuine out-of-fold) ---
oof = predictor.predict_proba_oof()
ag_oof_class = oof.values.argmax(1) + 1
ag_kappa = cohen_kappa_score(y, ag_oof_class, weights='linear')

print(f"\n{'='*58}")
print(f"HONEST MODEL — OOF linear-weighted kappa: {ag_kappa:.4f}")
print(f"{'='*58}")
print(predictor.leaderboard(silent=True)[['model','score_val']].to_string(index=False))
print("\nPer-class OOF report:")
print(classification_report(y, ag_oof_class,
      target_names=[f'ESI-{i}' for i in range(1,6)], digits=3))

# --- Test predictions + the submission file ---
ag_test_proba = predictor.predict_proba(ag_test).values
ag_test_pred  = ag_test_proba.argmax(1) + 1
submission = pd.DataFrame({'patient_id': test_eng['patient_id'], 'triage_acuity': ag_test_pred})
assert list(submission.columns) == list(sample_sub.columns), "column mismatch"
assert submission['triage_acuity'].between(1,5).all(),        "pred out of range"
assert len(submission) == len(sample_sub),                    "row count mismatch"
submission.to_csv('submission_honest.csv', index=False)
print(f"\n✓ submission_honest.csv written — {len(submission):,} rows")
print("Predicted vs train distribution (should be close):")
comp = pd.DataFrame({
    'predicted': submission['triage_acuity'].value_counts(normalize=True).sort_index(),
    'train':     y.value_counts(normalize=True).sort_index(),
}).round(3)
print(comp.to_string())

**Output read:** Honest **OOF linear-weighted kappa = 0.9114**. Per-class accuracy is near-perfect
for the urgent classes (ESI-1/2) and the errors concentrate at the clinically benign ESI-4/5 boundary.
The predicted test distribution closely tracks the training distribution, and `submission_honest.csv`
is the competition submission. Reaching the same score as an exhaustive AutoML search with a single
lean model is itself evidence the legitimate signal is simple and saturates quickly near ~0.91.


## Section 6 — Interpretability I: SHAP on the honest model

To explain the submitted model we fit a standalone LightGBM on identical features (AutoGluon selected
LightGBM with weight 1.0) and apply SHAP's exact `TreeExplainer`. We first **verify** the standalone
model reproduces the submitted model's OOF kappa, so the explanation faithfully represents the
submission.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 6 — Interpretability I: SHAP on the honest model
#   AutoGluon selected a single LightGBM (ensemble weight 1.0). We fit a
#   standalone LightGBM on identical features so SHAP's exact TreeExplainer can
#   be used, and VERIFY it reproduces the submitted model's OOF kappa — so the
#   SHAP explanation faithfully represents the submission. Contrast Section 4:
#   here the drivers are vitals/NEWS2, not text.
# ══════════════════════════════════════════════════════════════════════════
import lightgbm as lgb, shap, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

LGBM_PARAMS = dict(objective='multiclass', num_class=5, n_estimators=600,
                   learning_rate=0.05, max_depth=7, num_leaves=63,
                   subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
                   reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1)

# --- Verify the standalone LightGBM matches the submitted model (OOF kappa) ---
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof = np.zeros((len(X), 5))
for tr_i, va_i in skf.split(X, y):
    m = lgb.LGBMClassifier(**LGBM_PARAMS)
    m.fit(X.iloc[tr_i], y.iloc[tr_i]-1, categorical_feature=CAT_FEATURES,
          eval_set=[(X.iloc[va_i], y.iloc[va_i]-1)],
          callbacks=[lgb.early_stopping(40, verbose=False)])
    oof[va_i] = m.predict_proba(X.iloc[va_i])
proxy_kappa = cohen_kappa_score(y, oof.argmax(1)+1, weights='linear')
print(f"Standalone LightGBM OOF kappa: {proxy_kappa:.4f}  "
      f"(submitted AutoGluon model: {ag_kappa:.4f})  → faithful proxy ✓")

# --- Fit on full data for SHAP ---
lgb_full = lgb.LGBMClassifier(**LGBM_PARAMS)
lgb_full.fit(X, y - 1, categorical_feature=CAT_FEATURES)

expl = shap.TreeExplainer(lgb_full)
shap_sample = X.sample(min(3000, len(X)), random_state=SEED)
shap_raw = expl.shap_values(shap_sample)

# Normalise across SHAP versions: list of per-class arrays OR 3D ndarray
if isinstance(shap_raw, list):
    shap_by_class = shap_raw
else:
    shap_by_class = [shap_raw[:, :, c] for c in range(shap_raw.shape[-1])]
print(f"SHAP parsed into {len(shap_by_class)} per-class arrays, each {shap_by_class[0].shape}")

mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_by_class], axis=0)
imp = pd.Series(mean_abs, index=FEATURES).sort_values(ascending=False)
print("\nTop 15 features (mean |SHAP|, averaged across classes):")
print(imp.head(15).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
imp.head(15)[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('SHAP — honest model drivers (clinical, not text)', fontweight='bold')
ax.set_xlabel('mean |SHAP|')
plt.tight_layout(); plt.savefig('shap_importance.png', dpi=120, bbox_inches='tight'); plt.show()

plt.figure()
shap.summary_plot(shap_by_class[0], shap_sample, feature_names=FEATURES,
                  max_display=15, show=False)
plt.title('SHAP — drivers of ESI-1 (Immediate) prediction', fontweight='bold')
plt.tight_layout(); plt.savefig('shap_esi1_beeswarm.png', dpi=120, bbox_inches='tight'); plt.show()

**Output read:** The proxy reproduces the submission (≈0.908 vs 0.911), so its SHAP values are
representative. Globally, `pain_score` leads — but this reflects its role in separating the large
ESI-4/5 boundary. For the **critical** class (ESI-1), the beeswarm shows `gcs_total`, `spo2`, and
`respiratory_rate` driving the call in clinically correct directions. In other words, the feature that
sorts *volume* differs from the one that protects the *sickest* patients — and crucially, **none of
the top drivers is text**, in direct contrast to Section 4.


## Section 7 — Interpretability II: Glass-box EBM + Site-Consistency Audit

An Explainable Boosting Machine is *inherently* interpretable — its explanation is exact, not a
post-hoc approximation. We use it two ways: (a) as an independent check that the honest model relies
on clinical features, and (b) to **audit** whether a patient's *site* shifts predicted acuity when
clinical features are held constant — an equity check, reading per-site contributions directly off the
glass-box model with bagging uncertainty bounds.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 7 — Glass-box Model (EBM) + Site-Consistency Audit
#   An Explainable Boosting Machine: inherently interpretable, no post-hoc
#   approximation. It confirms the honest model relies on clinical features
#   (vitals, NEWS2) — the opposite of the text-dominated model in Section 4.
#   We also audit whether a patient's SITE shifts predicted acuity (an equity
#   check), reading per-site contributions directly off the glass-box model.
# ══════════════════════════════════════════════════════════════════════════
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import numpy as np, pandas as pd, matplotlib.pyplot as plt

EBM_KW = dict(
    random_state=SEED,
    interactions=0,            # main-effects only: fast + every term is a readable curve
    outer_bags=4,
    inner_bags=0,
    max_rounds=2000,
    early_stopping_rounds=50,
    n_jobs=-1,
)

# EBM wants categoricals as str, not pandas 'category' dtype
X_ebm = X.copy()
for c in CAT_FEATURES:
    X_ebm[c] = X_ebm[c].astype(str)

# ---- 6a: Glass-box EBM, 5-fold OOF kappa ----
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
ebm_oof = np.zeros(len(X_ebm))
for fold, (tr_i, va_i) in enumerate(skf.split(X_ebm, y)):
    ebm = ExplainableBoostingClassifier(**EBM_KW)
    ebm.fit(X_ebm.iloc[tr_i], y.iloc[tr_i])
    ebm_oof[va_i] = ebm.predict(X_ebm.iloc[va_i])
    print(f"  fold {fold+1}/5 done")
ebm_kappa = cohen_kappa_score(y, ebm_oof, weights='linear')
print(f"\nEBM (glass-box) OOF linear-weighted kappa: {ebm_kappa:.4f}")
print(f"  (vs honest black-box LightGBM {ag_kappa:.4f} — interpretability costs ~{ag_kappa-ebm_kappa:.3f})")

# Final EBM on all data for interpretation
ebm_full = ExplainableBoostingClassifier(**EBM_KW)
ebm_full.fit(X_ebm, y)

ebm_global = ebm_full.explain_global()
imp_ebm = pd.Series(ebm_global.data()['scores'],
                    index=ebm_global.data()['names']).sort_values(ascending=False)
print("\nEBM global importance (top 15):")
print(imp_ebm.head(15).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
imp_ebm.head(15)[::-1].plot.barh(ax=ax, color='seagreen')
ax.set_title('Honest glass-box model — clinical features drive predictions',
             fontweight='bold')
ax.set_xlabel('mean abs contribution')
plt.tight_layout(); plt.savefig('ebm_importance.png', dpi=120, bbox_inches='tight'); plt.show()

# ---- 6b: Site-consistency audit (full data) with bagging uncertainty bounds ----
X_audit = X_ebm.copy()
X_audit['site_id'] = train_raw['site_id'].astype(str).values
audit = ExplainableBoostingClassifier(**EBM_KW)
audit.fit(X_audit, y)

ag_ = audit.explain_global()
a_names = ag_.data()['names']
print("\n--- Site-consistency audit (full data) ---")
if 'site_id' in a_names:
    term   = ag_.data(a_names.index('site_id'))
    sites  = term['names']
    scores = np.asarray(term['scores'])          # (n_sites, n_classes)
    upper  = np.asarray(term['upper_bounds'])
    lower  = np.asarray(term['lower_bounds'])
    esi_levels = np.array(term['meta']['label_names'], dtype=float)

    def to_shift(arr): return (arr * esi_levels).sum(axis=1)
    centre     = to_shift(scores).mean()
    site_shift = to_shift(scores) - centre
    site_hi    = to_shift(upper)  - centre
    site_lo    = to_shift(lower)  - centre

    print("Per-site shift in predicted acuity (clinical features held constant):")
    print("(+ = toward LESS urgent, − = toward MORE urgent; 0 = network average)")
    order = np.argsort(site_shift)
    for i in order:
        print(f"    {sites[i]}: {site_shift[i]:+.4f}  "
              f"[{min(site_lo[i],site_hi[i]):+.4f}, {max(site_lo[i],site_hi[i]):+.4f}]")
    spread = site_shift.max() - site_shift.min()
    print(f"\n  Patients per site: {dict(train_raw['site_id'].value_counts())}")
    print(f"  → site-induced acuity spread: {spread:.4f} ESI units")

    out_i, nxt_i = order[-1], order[-2]
    out_low  = min(site_lo[out_i], site_hi[out_i])
    nxt_high = max(site_lo[nxt_i], site_hi[nxt_i])
    print(f"  Outlier {sites[out_i]} vs next site {sites[nxt_i]}: "
          f"{'SEPARATED (distinct)' if out_low > nxt_high else 'OVERLAPPING (not conclusive)'}")

    fig, ax = plt.subplots(figsize=(8, 4))
    yy  = np.arange(len(sites))
    err = np.abs(np.vstack([site_shift - site_lo, site_hi - site_shift]))[:, order]
    ax.barh(yy, site_shift[order], xerr=err, color='indianred', edgecolor='white', capsize=4)
    ax.set_yticks(yy); ax.set_yticklabels([sites[i] for i in order])
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Acuity shift (ESI units, vs network average)')
    ax.set_title('Site-level triage consistency audit (EBM bagging uncertainty bounds)',
                 fontweight='bold')
    plt.tight_layout(); plt.savefig('site_audit.png', dpi=120, bbox_inches='tight'); plt.show()
else:
    print("site_id term not found:", a_names[:10])

**Output read:** The glass-box EBM reaches **≈0.897 kappa** — within ~0.015 of the black-box model,
so interpretability costs very little here. Its feature ranking is all clinical (pain, respiratory
rate, GCS, SpO₂, NEWS2), independently agreeing with SHAP and again contrasting the text-driven leaky
model. The site audit shows a point-estimate spread of ~0.28 ESI units with one site (OUL-01) trending
toward lower acuity, **but the uncertainty bounds overlap** — so we report this as *not statistically
conclusive*. The contribution is the reusable audit method; on real ED data it could surface genuine
institutional inconsistency.


## Section 8 — Clinical Safety Analysis

Overall kappa weights all errors equally, but in triage they are not: **undertriage** (predicting
*less* urgent than truth) can be fatal, while **overtriage** mainly costs resources. This section
assesses safety in three parts: (8) the **undertriage profile** — rate, severity, and where errors
concentrate; (8b) **calibration** — whether the model's probabilities can be trusted, not just its
decisions; and (8c) **confidence intervals** — reporting the headline safety figures with bootstrap
bounds at publishable rigor. Together these answer the question a clinician actually asks: *can I
trust this tool not to send a sick patient to wait?*

### 8 — Undertriage profile
We measure undertriage and overtriage separately, quantify severe (≥2-level) undertriage of the
critically ill, and summarise with an illustrative asymmetric cost that penalises undertriage more heavily.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 8 — Cost-Sensitive Undertriage Analysis (clinical safety)
#   Overall kappa weights all errors equally, but in triage they are not equal:
#   undertriage (predicting LESS urgent than truth) can be fatal, while
#   overtriage merely costs resources. We quantify undertriage rate, severity,
#   and where it concentrates — the metrics a clinician actually cares about.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, matplotlib.pyplot as plt

y_true = y.values
# primary = honest submitted model's OOF predictions (from Section 5)
models = {'Honest model (submitted)': ag_oof_class}

def undertriage_report(name, y_pred):
    err  = y_pred - y_true                  # >0 = undertriage (less urgent), <0 = overtriage
    under, over = err > 0, err < 0
    crit = np.isin(y_true, [1, 2])          # patients who must not be missed
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(f"  Exact accuracy                       : {(err==0).mean():.1%}")
    print(f"  Undertriage rate (any)               : {under.mean():.1%}")
    print(f"  Overtriage rate  (any)               : {over.mean():.1%}")
    print(f"  Undertriage:Overtriage ratio         : {under.sum()/max(over.sum(),1):.2f}")
    print(f"  Undertriage among true ESI-1/2 (DANGEROUS): {under[crit].mean():.1%}")
    print(f"  Severe undertriage (≥2 levels too low)     : {(err>=2).mean():.2%}")
    print(f"  Severe undertriage among true ESI-1/2      : {(err[crit]>=2).mean():.2%}")

for name, yp in models.items():
    undertriage_report(name, yp)

# --- Error direction by true ESI level ---
err = ag_oof_class - y_true
rows = []
for lvl in range(1, 6):
    m = y_true == lvl
    rows.append({'ESI': f'ESI-{lvl}',
                 'undertriage_%': (err[m] > 0).mean()*100,
                 'exact_%'      : (err[m] == 0).mean()*100,
                 'overtriage_%' : (err[m] < 0).mean()*100})
by_level = pd.DataFrame(rows)
print("\nError direction by true ESI level (honest model OOF):")
print(by_level.round(1).to_string(index=False))

# --- Figures: error direction by level + asymmetric cost ---
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
b = by_level.set_index('ESI')[['undertriage_%','exact_%','overtriage_%']]
b.plot(kind='bar', stacked=True, ax=ax[0],
       color=['#d62728','#2ca02c','#ff7f0e'], edgecolor='white')
ax[0].set_title('Error direction by true ESI', fontweight='bold')
ax[0].set_ylabel('% of patients'); ax[0].legend(loc='lower right')
ax[0].set_xticklabels(b.index, rotation=0)

# Asymmetric cost: undertriage penalised more than overtriage (weights illustrative)
def cost(y_pred, w_under=3.0, w_over=1.0):
    d = y_pred - y_true
    return np.where(d > 0, w_under*d**2, w_over*d**2).mean()
c = cost(ag_oof_class)
ax[1].bar(['Honest model'], [c], color='steelblue', edgecolor='white')
ax[1].text(0, c, f'{c:.3f}', ha='center', va='bottom')
ax[1].set_title('Asymmetric triage cost (undertriage × 3)', fontweight='bold')
ax[1].set_ylabel('mean cost (lower = better)')
plt.tight_layout(); plt.savefig('undertriage_analysis.png', dpi=120, bbox_inches='tight'); plt.show()

print(f"\nAsymmetric cost (undertriage penalised 3×): {c:.3f}")
print("Note: cost weights are illustrative; real weights would be set by clinical consensus.")

**Output read — the clinical headline:** among truly critical patients (ESI-1/2), **severe
undertriage (≥2 levels too low) is 0.00%** — the model never makes a catastrophic miss on a sick
patient across 80,000 cases. Undertriage and overtriage are essentially balanced (ratio ≈1.0, no hidden
bias), and the model is most accurate on ESI-2, the most safety-critical undertriageable class. Errors
concentrate at the benign ESI-4/5 boundary. (Note: ESI-1 cannot be overtriaged and ESI-5 cannot be
undertriaged, so those boundary rates are partly structural, not model failures.) Cost weights are
illustrative and would be set by clinical consensus in practice.


### 8b — Calibration
A probability is only useful for decision support if it is *calibrated*: when the model says "70%
chance critical", roughly 70% of such patients should truly be critical. We assess calibration of the
clinically key signal — the predicted probability that a patient is critical (ESI-1/2) — one-vs-rest,
using the honest model's out-of-fold probabilities, alongside a per-class view.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 8b — Calibration Analysis
#   For decision support, a probability is only useful if it is calibrated:
#   when the model says "70% chance critical", ~70% of such patients should be
#   critical. We assess calibration of the clinically key signal — the predicted
#   probability that a patient is critical (ESI-1 or ESI-2) — one-vs-rest, using
#   the honest model's out-of-fold probabilities.
# ══════════════════════════════════════════════════════════════════════════
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import numpy as np, matplotlib.pyplot as plt

# OOF probabilities from the honest model (Section 5). `oof` may be a DataFrame
# (from predict_proba_oof) or an array — normalise to a numpy array, cols = ESI1..5.
oof_arr = oof.values if hasattr(oof, 'values') else np.asarray(oof)

# Critical = ESI-1 or ESI-2  ->  P(critical) = p1 + p2
p_critical = oof_arr[:, 0] + oof_arr[:, 1]
y_critical = np.isin(y.values, [1, 2]).astype(int)

frac_pos, mean_pred = calibration_curve(y_critical, p_critical, n_bins=10, strategy='quantile')
brier = brier_score_loss(y_critical, p_critical)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

# Reliability curve (critical vs rest)
ax[0].plot([0, 1], [0, 1], '--', color='gray', lw=1, label='perfect calibration')
ax[0].plot(mean_pred, frac_pos, 'o-', color='#1d9e75', lw=2, label='honest model')
ax[0].set_xlabel('Mean predicted P(critical)')
ax[0].set_ylabel('Observed fraction critical')
ax[0].set_title(f'Reliability — critical-class (ESI 1/2) detection\nBrier score = {brier:.4f}',
                fontweight='bold')
ax[0].legend(loc='upper left'); ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1)

# Per-class reliability (one-vs-rest) as a secondary view
for cls in range(5):
    yc = (y.values == cls + 1).astype(int)
    fp, mp = calibration_curve(yc, oof_arr[:, cls], n_bins=10, strategy='quantile')
    ax[1].plot(mp, fp, 'o-', lw=1.3, markersize=4, label=f'ESI-{cls+1}')
ax[1].plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax[1].set_xlabel('Mean predicted probability')
ax[1].set_ylabel('Observed frequency')
ax[1].set_title('Per-class reliability (one-vs-rest)', fontweight='bold')
ax[1].legend(loc='upper left', fontsize=9); ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)

plt.tight_layout(); plt.savefig('calibration.png', dpi=120, bbox_inches='tight'); plt.show()

print(f"Brier score (critical-class detection): {brier:.4f}  (lower is better; 0 = perfect)")
print("Interpretation: points near the diagonal mean the predicted probabilities are")
print("trustworthy — e.g. patients given ~70% critical probability are critical ~70% of the time.")

**Output read:** The model is **well-calibrated** — the critical-class reliability curve sits almost
exactly on the diagonal with a **Brier score of 0.0034** (0 = perfect), and all five per-class curves
track the diagonal closely. This means the predicted probabilities are trustworthy, not just the
argmax decisions — a prerequisite for decision-support use, where a clinician may act on the *degree*
of urgency. (As with overall accuracy, this strong calibration partly reflects clean synthetic data and
would require re-validation on real ED data.)

### 8c — Confidence intervals on safety statistics
The headline safety figures are point estimates over 80,000 out-of-fold cases. We attach 95% bootstrap
confidence intervals so the claims are reported at publishable rigor. For the zero-event statistic
(severe undertriage of critical patients), where the bootstrap cannot widen a true-zero count, we
additionally report the analytic *rule-of-three* upper bound.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 8c — Confidence Intervals on Safety Statistics
#   The headline safety figures (undertriage rates, the undertriage:overtriage
#   balance, severe-undertriage of critical patients) are point estimates over
#   80k OOF cases. We attach 95% bootstrap confidence intervals so the claims
#   are reported at publishable rigor rather than as bare numbers.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np

rng = np.random.default_rng(SEED)
yt   = y.values
err  = ag_oof_class - yt              # >0 undertriage, <0 overtriage
crit = np.isin(yt, [1, 2])
n    = len(yt)
B    = 2000                            # bootstrap resamples

def _stats(idx):
    e  = err[idx]; c = crit[idx]
    under = e > 0; over = e < 0
    ec = e[c]
    return {
        'undertriage_rate'      : under.mean(),
        'overtriage_rate'       : over.mean(),
        'under_over_ratio'      : under.sum() / max(over.sum(), 1),
        'crit_undertriage_rate' : (ec > 0).mean() if c.any() else np.nan,
        'crit_severe_under_rate': (ec >= 2).mean() if c.any() else np.nan,
    }

point = _stats(np.arange(n))
keys  = list(point.keys())
boot  = {k: [] for k in keys}
for _ in range(B):
    idx = rng.integers(0, n, n)        # resample with replacement
    s = _stats(idx)
    for k in keys: boot[k].append(s[k])

print(f"95% bootstrap confidence intervals ({B} resamples, n={n:,}):\n")
labels = {
    'undertriage_rate'      : 'Undertriage rate (any)',
    'overtriage_rate'       : 'Overtriage rate (any)',
    'under_over_ratio'      : 'Undertriage : Overtriage ratio',
    'crit_undertriage_rate' : 'Undertriage of critical (ESI-1/2)',
    'crit_severe_under_rate': 'SEVERE undertriage of critical (≥2 levels)',
}
for k in keys:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    pe = point[k]
    if k == 'under_over_ratio':
        print(f"  {labels[k]:<46s}: {pe:.3f}   [{lo:.3f}, {hi:.3f}]")
    else:
        print(f"  {labels[k]:<46s}: {pe*100:.2f}%  [{lo*100:.2f}%, {hi*100:.2f}%]")

print("\nNote: the severe-undertriage-of-critical estimate is 0%; its upper bound quantifies")
print("the worst case consistent with the data at 95% confidence.")

# Rule of three: with 0 observed events in N trials, the 95% upper bound ≈ 3/N.
# Bootstrap cannot widen a true-zero count, so we report this analytic bound instead.
n_crit = int(crit.sum())
print(f"\nSevere undertriage of critical: 0 events in {n_crit:,} patients.")
print(f"  Rule-of-three 95% upper bound ≈ {3/n_crit*100:.3f}%  (zero observed ≠ zero risk).")

**Output read:** The intervals are tight given the large sample. Notably, the undertriage:overtriage
ratio is **1.014 with a 95% CI of [0.970, 1.061]** — the interval includes 1.0, so there is **no
statistically significant directional bias**. Severe undertriage of critical patients was observed in
**0 of 16,661** such patients; since zero observed events does not mean zero risk, the rule-of-three
95% upper bound is **≈ 0.018%**. This is the rigorous form of the safety headline: not merely "zero
events", but zero events with a quantified worst-case bound.

## Summary

| Model | OOF linear-weighted kappa | Status |
|---|---|---|
| Text-only (TF-IDF) | ~0.999 | **Leakage — not submitted** |
| Honest + TF-IDF | ~0.999 | **Leakage — not submitted** |
| **Honest model (LightGBM)** | **0.9114** | **Submitted** |
| Glass-box EBM | ~0.897 | Interpretability check |

**Key takeaways**
- The synthetic chief-complaint text encodes the label: text alone predicts acuity at ~0.999 and
  99.6% of repeated complaints map to a single acuity. Because the text is near-deterministic of the
  label, vectorizing it inflates scores to ~0.99 regardless of the downstream classifier — this is
  leakage of the data-generating process, not clinical signal.
- The honest performance ceiling is ~0.91, bounded by irreducible label noise.
- The submitted model is interpretable (SHAP + EBM independently agree on clinical drivers),
  **well-calibrated** (Brier 0.0034 for critical-class detection), and clinically safe: **zero severe
  undertriage of critical patients** (0 / 16,661; rule-of-three 95% upper bound ≈ 0.018%), with no
  statistically significant directional bias (undertriage:overtriage ratio 95% CI includes 1.0).
- A site-level equity audit method is demonstrated; results here are not statistically conclusive.

*Limitations:* results are on synthetic data calibrated to published ED literature. The text-leakage
finding and the strong calibration are properties of this dataset's generation and would require
re-validation on real ED data — where, by contrast, careful NLP of chief complaints would likely help.
The optional LLM narrative layer is provided for bedside readability only and does not affect predictions.

## Section 9 — Plain-Language Triage Rationales (optional decision-support layer)

The model outputs an ESI number; a triage nurse benefits from a short, plain-language
explanation of *why*. This optional layer turns each prediction into a 2–3 sentence
clinical rationale, grounded in the patient's actual vitals and the factors the model
weighted. **The LLM does not make or alter the triage decision** — it only explains the
prediction already produced by the validated model in Section 5.

The layer tries an LLM backend in priority order (OpenAI → Anthropic direct → AWS Bedrock).
If no credentials are configured, it falls back to a **deterministic rule-based explanation**
built from the same inputs — so the notebook always runs end-to-end and remains fully
reproducible without any secrets.

> **Reproducibility & honesty note:** The narratives shown here were generated and tested
> using **AWS Bedrock (Claude Sonnet 4)** via an account-specific inference profile. When run
> without AWS credentials (e.g. by reviewers, or with internet disabled), this section
> automatically uses the rule-based fallback. The OpenAI and Anthropic-direct code paths are
> provided for convenience but were **not** tested by the author.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 9A — Narrative layer setup (optional)
#   Tries an LLM in priority order: OpenAI → Anthropic direct → AWS Bedrock
#   (Claude). If none is configured, falls back to a deterministic rule-based
#   explanation so the notebook ALWAYS runs end-to-end without secrets.
#
#   DISCLAIMER: This notebook's narrative output was generated and tested ONLY
#   via the AWS Bedrock (Claude) path. The OpenAI and Anthropic-direct paths are
#   provided for convenience but were NOT tested by the author.
# ══════════════════════════════════════════════════════════════════════════
import os

LLM_MODE = 'fallback'        # 'openai' | 'claude' | 'bedrock' | 'fallback'
llm_client = None
LLM_MODEL_NAME = None

def _get_secret(name):
    """Read a Kaggle secret (tolerant of stray spaces); None if absent."""
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for cand in (name, name + ' ', name.strip()):
            try:
                v = client.get_secret(cand)
                if v: return v
            except Exception:
                continue
        return None
    except Exception:
        return os.environ.get(name)

# --- 1) OpenAI (UNTESTED path) ---
_openai_key = _get_secret('OPENAI_API_KEY')
if _openai_key and LLM_MODE == 'fallback':
    try:
        from openai import OpenAI
        llm_client = OpenAI(api_key=_openai_key)
        LLM_MODEL_NAME = 'gpt-4o-mini'
        LLM_MODE = 'openai'
        print(f"✓ LLM: OpenAI ready ({LLM_MODEL_NAME}) [untested path]")
    except Exception as e:
        print(f"  OpenAI init failed: {e}")

# --- 2) Anthropic direct (UNTESTED path) ---
_anthropic_key = _get_secret('ANTHROPIC_API_KEY')
if _anthropic_key and LLM_MODE == 'fallback':
    try:
        import anthropic
        llm_client = anthropic.Anthropic(api_key=_anthropic_key)
        LLM_MODEL_NAME = 'claude-sonnet-4-6'
        LLM_MODE = 'claude'
        print(f"✓ LLM: Claude (direct API) ready ({LLM_MODEL_NAME}) [untested path]")
    except Exception as e:
        print(f"  Anthropic init failed: {e}")

# --- 3) AWS Bedrock — Claude (TESTED path) ---
_aws_key    = _get_secret('AWS_ACCESS_KEY_ID')
_aws_secret = _get_secret('AWS_SECRET_ACCESS_KEY')
_aws_region = _get_secret('AWS_REGION') or 'us-east-1'
if _aws_key and _aws_secret and LLM_MODE == 'fallback':
    try:
        import boto3
        llm_client = boto3.client(
            'bedrock-runtime',
            region_name=_aws_region,
            aws_access_key_id=_aws_key,
            aws_secret_access_key=_aws_secret,
        )
        # Bedrock Claude model ID — adjust to the model enabled in your account
        LLM_MODEL_NAME = 'us.anthropic.claude-sonnet-4-20250514-v1:0'
        LLM_MODE = 'bedrock'
        print(f"✓ LLM: AWS Bedrock Claude ready ({LLM_MODEL_NAME}) [tested path]")
    except Exception as e:
        print(f"  Bedrock init failed: {e}")

if LLM_MODE == 'fallback':
    print("ℹ No LLM configured — using deterministic rule-based explanations.")
    print("  (Keeps the notebook fully reproducible without any credentials.)")

### 9A — Backend setup · 9B — Generation

Section 9A resolves which backend is available and prints the active mode; Section 9B builds
a grounded prompt (vitals + assigned acuity + top weighted factors) and generates a rationale
for a low/mid/high-acuity sample of test patients. With a working LLM backend it narrates five
patients (one per acuity tier); in fallback mode it narrates three.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 9B — Plain-language triage rationales
#   Explains the model's prediction (does not make/alter it). Routes to whichever
#   backend 9A configured; if none, a deterministic template produces a sensible
#   clinical explanation from the same inputs.
# ══════════════════════════════════════════════════════════════════════════
import json
ESI_LABEL = {1:'ESI-1 (Immediate / resuscitation)', 2:'ESI-2 (Emergent / high-risk)',
             3:'ESI-3 (Urgent)', 4:'ESI-4 (Less urgent)', 5:'ESI-5 (Non-urgent)'}
cc_lookup = cc.set_index('patient_id')['chief_complaint_raw'].to_dict()

PROMPT_TMPL = """You are a clinical decision-support assistant for emergency department triage.
A machine-learning model has already assigned this patient a triage acuity level.
Explain the model's reasoning in 2-3 sentences of plain clinical language for a triage nurse.
Do NOT change the level, do NOT add diagnoses, and do NOT invent findings not listed.

PATIENT VITALS AT TRIAGE:
{vitals}

CHIEF COMPLAINT: "{complaint}"

MODEL'S ASSIGNED ACUITY: {label}

TOP FACTORS THE MODEL WEIGHTED:
{drivers}

End with one line: "Model confidence note:" reminding the nurse this is decision support."""

def _rule_based(vitals, label, drivers, complaint):
    drv = "; ".join(drivers)
    return (f"The model assigned {label} based on: {drv}. "
            f"Recorded vitals: {', '.join(f'{k} {v}' for k,v in vitals.items() if v is not None)}. "
            f'Chief complaint: "{complaint}". '
            f"Model confidence note: automated decision support, not a substitute for clinical judgment.")

def generate_narrative(vitals, label, drivers, complaint, max_tokens=220):
    prompt = PROMPT_TMPL.format(
        vitals="\n".join(f"  - {k}: {v}" for k, v in vitals.items()),
        complaint=complaint, label=label,
        drivers="\n".join(f"  - {d}" for d in drivers))
    try:
        if LLM_MODE == 'openai':
            r = llm_client.chat.completions.create(
                model=LLM_MODEL_NAME, max_tokens=max_tokens, temperature=0.3,
                messages=[{"role":"user","content":prompt}])
            return r.choices[0].message.content.strip()
        if LLM_MODE == 'claude':
            r = llm_client.messages.create(
                model=LLM_MODEL_NAME, max_tokens=max_tokens, temperature=0.3,
                messages=[{"role":"user","content":prompt}])
            return r.content[0].text.strip()
        if LLM_MODE == 'bedrock':
            body = json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": max_tokens, "temperature": 0.3,
                "messages": [{"role":"user","content":prompt}],
            })
            resp = llm_client.invoke_model(modelId=LLM_MODEL_NAME, body=body)
            payload = json.loads(resp['body'].read())
            return payload['content'][0]['text'].strip()
    except Exception as e:
        print(f"  (LLM call failed: {e} — using rule-based fallback)")
    return _rule_based(vitals, label, drivers, complaint)

n_demo = 5 if LLM_MODE != 'fallback' else 3
idx_sorted = np.argsort(ag_test_pred)
demo_idx = [int(i) for i in idx_sorted[np.linspace(0, len(idx_sorted)-1, n_demo).astype(int)]]

print(f"Narrative mode: {LLM_MODE}\n")
for rank, i in enumerate(demo_idx, 1):
    pid  = test_eng['patient_id'].iloc[i]; pred = int(ag_test_pred[i]); row = test_eng.iloc[i]
    vitals = {'age':row.get('age'),'GCS':row.get('gcs_total'),'systolic_bp':row.get('systolic_bp'),
              'heart_rate':row.get('heart_rate'),'resp_rate':row.get('respiratory_rate'),
              'SpO2':row.get('spo2'),'temp_C':row.get('temperature_c'),
              'NEWS2':row.get('news2_score'),'mental_status':row.get('mental_status_triage')}
    drivers = []
    if row.get('gcs_total',15) < 14: drivers.append(f"reduced GCS ({row.get('gcs_total')})")
    if row.get('news2_score',0) >= 5: drivers.append(f"elevated NEWS2 ({row.get('news2_score')})")
    if row.get('spo2',100) < 94: drivers.append(f"low SpO2 ({row.get('spo2')}%)")
    if row.get('respiratory_rate',16) > 22: drivers.append(f"tachypnea (RR {row.get('respiratory_rate')})")
    if not drivers: drivers = ["vitals within or near normal range","no high-risk complaint flags"]
    complaint = str(cc_lookup.get(pid,"")).replace('\uff0c',',')[:200]

    print(f"{'='*70}\nDEMO PATIENT {rank} (id={pid}) — model says {ESI_LABEL[pred]}\n{'='*70}")
    print(generate_narrative(vitals, ESI_LABEL[pred], drivers, complaint), "\n")

**Output read:** With the AWS Bedrock backend active, the layer produces clinically coherent
rationales that correctly reference each patient's actual vitals and reasoning for the assigned
tier — e.g. for ESI-1, citing reduced GCS, elevated NEWS2, and low SpO₂ as multi-system
instability; for ESI-4/5, noting stable vitals and the absence of high-risk flags. The narratives
faithfully explain the model's output without overriding it, and every response ends with a
decision-support disclaimer. Crucially, the explanation is *post-hoc readability* — accuracy
comes entirely from the tabular model (Section 5), and the interpretable feature attributions
(Sections 6–7) remain the rigorous basis for trust. In the absence of credentials, the
deterministic fallback still surfaces the prediction, the weighted factors, and the vitals.